# 🏠 HGT Property Valuation - TPU Training on Colab

This notebook trains a **Heterogeneous Graph Transformer (HGT)** for Bangkok property valuation using Google Colab TPU.

## Pipeline Overview
1. **Setup** - Install PyTorch/XLA and dependencies
2. **Data Upload** - Upload pre-built graph data
3. **Model Definition** - HGT with cold-start handling
4. **Training** - TPU-accelerated training loop
5. **Evaluation** - Metrics and visualization
6. **Export** - Download trained model

---

⚠️ **Before running:** Go to `Runtime → Change runtime type → TPU`

## 1. Environment Setup

In [19]:
# Check TPU/GPU availability
import os
import sys

# Detect runtime type (Colab TPU v2/v4/v5)
if 'TPU_ACCELERATOR_TYPE' in os.environ or 'COLAB_TPU_ADDR' in os.environ:
    print("✅ TPU detected")
    DEVICE_TYPE = 'tpu'
elif os.path.exists('/dev/nvidia0') or 'CUDA_VISIBLE_DEVICES' in os.environ:
    print("✅ GPU detected")
    DEVICE_TYPE = 'gpu'
else:
    print("⚠️ No accelerator detected, using CPU")
    DEVICE_TYPE = 'cpu'

print(f"Runtime: {DEVICE_TYPE.upper()}")

✅ TPU detected
Runtime: TPU


In [20]:
# Install dependencies
# PyTorch/XLA for TPU support (use Colab's pre-installed versions when possible)

if DEVICE_TYPE == 'tpu':
    # PyG with matching torch version - don't need scatter/sparse for HGTConv
    !pip install -q torch-geometric 2>/dev/null
elif DEVICE_TYPE == 'gpu':
    !pip install -q torch-geometric torch-scatter torch-sparse 2>/dev/null
else:
    !pip install -q torch torch-geometric 2>/dev/null

!pip install -q numpy pandas scikit-learn matplotlib tqdm 2>/dev/null

print("✅ Dependencies installed")
print("Note: torch-scatter/torch-sparse warnings can be ignored - PyG uses native fallbacks")

✅ Dependencies installed
Note: torch-scatter/torch-sparse warnings can be ignored - PyG uses native fallbacks


In [21]:
# Import libraries
import json
import logging
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

# PyTorch Geometric
from torch_geometric.data import HeteroData
from torch_geometric.nn import HGTConv

# XLA for TPU (conditional)
if DEVICE_TYPE == 'tpu':
    import torch_xla
    # Use new API if available, fallback to legacy
    if hasattr(torch_xla, 'device'):
        DEVICE = torch_xla.device()
    else:
        import torch_xla.core.xla_model as xm
        DEVICE = xm.xla_device()
    print(f"✅ XLA device: {DEVICE}")
elif DEVICE_TYPE == 'gpu':
    DEVICE = torch.device('cuda')
    print(f"✅ CUDA device: {torch.cuda.get_device_name(0)}")
else:
    DEVICE = torch.device('cpu')
    print("Using CPU")

# Import xm for TPU operations
if DEVICE_TYPE == 'tpu':
    import torch_xla.core.xla_model as xm

# Check PyTorch version
print(f"PyTorch version: {torch.__version__}")

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

✅ XLA device: xla:0
PyTorch version: 2.9.0+cpu


## 2. Data Upload

Upload the pre-built heterogeneous graph (`hetero_graph.pt`).

### Option A: Google Drive (Recommended)
1. Upload `hetero_graph.pt` to your Google Drive
2. Run cell 7 to mount Drive and set path

### Option B: Direct Upload (Colab Browser Only)
- Only works when running directly in Colab browser
- Skip cell 7, run cell 7b instead

**To generate this file locally:**
```bash
cd gis-server
python -m scripts.build_h3_features
python -m scripts.train_hex2vec
python -m scripts.build_hetero_graph
```

In [23]:
# Option A: Load from Google Drive (RECOMMENDED)
# 1. First, upload hetero_graph.pt to your Google Drive
# 2. Then run this cell

from google.colab import drive
drive.mount('/content/drive')

# Set the path to your file in Google Drive
# Adjust this path based on where you uploaded the file
GRAPH_PATH = '/content/drive/MyDrive/hetero_graph.pt'

# Alternative paths - uncomment the one that matches your Drive structure:
# GRAPH_PATH = '/content/drive/MyDrive/Colab Notebooks/hetero_graph.pt'
# GRAPH_PATH = '/content/drive/MyDrive/data/hetero_graph.pt'

import os
if os.path.exists(GRAPH_PATH):
    print(f"✅ Found graph at: {GRAPH_PATH}")
else:
    print(f"❌ File not found at: {GRAPH_PATH}")
    print("Please upload hetero_graph.pt to your Google Drive and update GRAPH_PATH")

ValueError: mount failed

In [ ]:
# # Option B: Direct Upload (Only works in Colab browser)
# # Skip this cell if using Option A (Google Drive)

# from google.colab import files

# print("📤 Upload your hetero_graph.pt file:")
# uploaded = files.upload()

# GRAPH_PATH = list(uploaded.keys())[0]
# print(f"✅ Uploaded: {GRAPH_PATH}")

In [ ]:
# Load graph
data = torch.load(GRAPH_PATH, map_location='cpu', weights_only=False)

print("\n📊 Graph Structure:")
print(f"Node types: {data.node_types}")
print(f"Edge types: {data.edge_types}")

for node_type in data.node_types:
    if hasattr(data[node_type], 'x'):
        print(f"  {node_type}: {data[node_type].x.shape[0]} nodes, {data[node_type].x.shape[1]} features")

for edge_type in data.edge_types:
    if hasattr(data[edge_type], 'edge_index'):
        print(f"  {edge_type}: {data[edge_type].edge_index.shape[1]} edges")

# Check for target labels
if hasattr(data['property'], 'y'):
    print(f"\n📈 Target (property prices):")
    print(f"   Range: {data['property'].y.min():,.0f} - {data['property'].y.max():,.0f} THB")
    print(f"   Mean: {data['property'].y.mean():,.0f} THB")
else:
    print("⚠️ No target labels found in data['property'].y")

## 3. Model Definition

HGT with meta-relation attention for heterogeneous property graphs.

In [ ]:
class PropertyEncoder(nn.Module):
    """Encode property intrinsic features."""
    
    def __init__(self, input_dim: int, hidden_dim: int, dropout: float = 0.1):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return self.norm(x)


class NodeTypeEncoder(nn.Module):
    """Generic encoder for auxiliary node types."""
    
    def __init__(self, input_dim: int, hidden_dim: int):
        super().__init__()
        self.fc = nn.Linear(input_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.relu(self.fc(x))
        return self.norm(x)

In [ ]:
class HGTValuator(nn.Module):
    """
    Heterogeneous Graph Transformer for Property Valuation.
    
    Architecture:
    1. Node-type-specific encoders project features to common hidden dim
    2. HGTConv layers perform meta-relation attention
    3. Property embeddings through MLP head for price prediction
    """

    def __init__(
        self,
        node_feature_dims: dict,
        hidden_dim: int = 128,
        num_heads: int = 4,
        num_layers: int = 2,
        dropout: float = 0.1,
        metadata: tuple = None,
    ):
        super().__init__()
        
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.metadata = metadata
        
        # Node-type-specific encoders
        self.encoders = nn.ModuleDict()
        for node_type, input_dim in node_feature_dims.items():
            if node_type == 'property':
                self.encoders[node_type] = PropertyEncoder(input_dim, hidden_dim, dropout)
            else:
                self.encoders[node_type] = NodeTypeEncoder(input_dim, hidden_dim)
        
        # HGT Convolution layers
        self.convs = nn.ModuleList()
        for _ in range(num_layers):
            conv = HGTConv(
                in_channels=hidden_dim,
                out_channels=hidden_dim,
                metadata=metadata,
                heads=num_heads,
            )
            self.convs.append(conv)
        
        # Layer norms for residual connections
        self.norms = nn.ModuleList([
            nn.LayerNorm(hidden_dim) for _ in range(num_layers)
        ])
        
        self.dropout = nn.Dropout(dropout)
        
        # Prediction head
        self.prediction_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, hidden_dim // 4),
            nn.ReLU(),
            nn.Linear(hidden_dim // 4, 1),
        )
        
        # Cold-start aggregator
        self.cold_start_aggregator = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )

    def forward(
        self,
        x_dict: dict,
        edge_index_dict: dict,
        cold_start_mask: torch.Tensor = None,
    ) -> torch.Tensor:
        # Encode each node type
        h_dict = {}
        for node_type, x in x_dict.items():
            if node_type in self.encoders:
                h_dict[node_type] = self.encoders[node_type](x)
            else:
                # Fallback: project to hidden dim
                h_dict[node_type] = F.relu(nn.Linear(x.size(-1), self.hidden_dim).to(x.device)(x))
        
        # HGT message passing
        for i, conv in enumerate(self.convs):
            h_dict_new = conv(h_dict, edge_index_dict)
            
            # Residual connection + layer norm
            for node_type in h_dict:
                if node_type in h_dict_new:
                    h_dict[node_type] = self.norms[i](
                        h_dict[node_type] + self.dropout(h_dict_new[node_type])
                    )
        
        # Get property embeddings
        property_emb = h_dict['property']
        
        # Cold-start handling: aggregate from neighbors for new cells
        if cold_start_mask is not None and cold_start_mask.any():
            # Average pooling for cold-start nodes
            cold_emb = self.cold_start_aggregator(property_emb[cold_start_mask])
            property_emb = property_emb.clone()
            property_emb[cold_start_mask] = cold_emb
        
        # Predict prices
        prices = self.prediction_head(property_emb).squeeze(-1)
        
        return prices

In [ ]:
class ValuationLoss(nn.Module):
    """
    Combined loss for property valuation.
    
    Uses log-space MSE for scale-invariant learning,
    plus a cold-start penalty term.
    """
    
    def __init__(self, cold_start_weight: float = 0.7):
        super().__init__()
        self.cold_start_weight = cold_start_weight

    def forward(
        self,
        predictions: torch.Tensor,
        targets: torch.Tensor,
        cold_start_mask: torch.Tensor = None,
    ) -> torch.Tensor:
        # Log-space MSE for scale invariance
        log_pred = torch.log1p(predictions.clamp(min=0))
        log_target = torch.log1p(targets.clamp(min=0))
        
        mse_loss = F.mse_loss(log_pred, log_target, reduction='none')
        
        # Apply lower weight to cold-start nodes (noisier labels)
        if cold_start_mask is not None:
            weights = torch.ones_like(mse_loss)
            weights[cold_start_mask] = self.cold_start_weight
            mse_loss = mse_loss * weights
        
        return mse_loss.mean()

In [ ]:
def create_model_from_data(data: HeteroData, hidden_dim: int = 128, num_heads: int = 4, num_layers: int = 2) -> HGTValuator:
    """Create HGT model matching the graph structure."""
    
    # Get feature dimensions for each node type
    node_feature_dims = {}
    for node_type in data.node_types:
        if hasattr(data[node_type], 'x'):
            node_feature_dims[node_type] = data[node_type].x.shape[1]
    
    # Get graph metadata
    metadata = data.metadata()
    
    model = HGTValuator(
        node_feature_dims=node_feature_dims,
        hidden_dim=hidden_dim,
        num_heads=num_heads,
        num_layers=num_layers,
        metadata=metadata,
    )
    
    print(f"✅ Created HGT model:")
    print(f"   Node types: {list(node_feature_dims.keys())}")
    print(f"   Hidden dim: {hidden_dim}, Heads: {num_heads}, Layers: {num_layers}")
    print(f"   Parameters: {sum(p.numel() for p in model.parameters()):,}")
    
    return model

## 4. Training Configuration

In [ ]:
# Hyperparameters
CONFIG = {
    'hidden_dim': 128,
    'num_heads': 4,
    'num_layers': 2,
    'learning_rate': 1e-3,
    'weight_decay': 1e-4,
    'epochs': 100,
    'patience': 15,  # Early stopping
    'train_ratio': 0.8,
    'val_ratio': 0.1,
}

print("📋 Training Configuration:")
for k, v in CONFIG.items():
    print(f"   {k}: {v}")

In [ ]:
# Validate graph data has required attributes
def validate_graph(data):
    """Check graph has all required attributes for training."""
    errors = []
    
    # Check property node features
    if not hasattr(data['property'], 'x') or data['property'].x is None:
        errors.append("Missing property node features (data['property'].x)")
    
    # Check target labels
    if not hasattr(data['property'], 'y') or data['property'].y is None:
        errors.append("Missing target labels (data['property'].y)")
    
    # Check edge indices
    for edge_type in data.edge_types:
        if not hasattr(data[edge_type], 'edge_index'):
            errors.append(f"Missing edge_index for {edge_type}")
    
    if errors:
        print("❌ Graph validation failed:")
        for e in errors:
            print(f"   - {e}")
        raise ValueError("Graph missing required attributes")
    else:
        print("✅ Graph validation passed")
        
validate_graph(data)

In [ ]:
def prepare_data_splits(data: HeteroData, train_ratio: float, val_ratio: float) -> HeteroData:
    """Create train/val/test masks with stratified cold-start distribution."""
    
    num_properties = data['property'].x.size(0)
    indices = np.arange(num_properties)
    
    # Get cold-start mask (handle case where it doesn't exist)
    if hasattr(data['property'], 'cold_start_mask'):
        cold_start = data['property'].cold_start_mask.numpy()
        # Only stratify if we have both cold and warm nodes
        use_stratify = cold_start.any() and (~cold_start).any()
    else:
        cold_start = np.zeros(num_properties, dtype=bool)
        use_stratify = False
    
    # Stratified split (only if meaningful stratification possible)
    stratify_arg = cold_start if use_stratify else None
    
    train_idx, temp_idx = train_test_split(
        indices, train_size=train_ratio, stratify=stratify_arg, random_state=42
    )
    
    val_size = val_ratio / (1 - train_ratio)
    stratify_temp = cold_start[temp_idx] if use_stratify else None
    val_idx, test_idx = train_test_split(
        temp_idx, train_size=val_size, stratify=stratify_temp, random_state=42
    )
    
    # Create masks
    train_mask = torch.zeros(num_properties, dtype=torch.bool)
    val_mask = torch.zeros(num_properties, dtype=torch.bool)
    test_mask = torch.zeros(num_properties, dtype=torch.bool)
    
    train_mask[train_idx] = True
    val_mask[val_idx] = True
    test_mask[test_idx] = True
    
    data['property'].train_mask = train_mask
    data['property'].val_mask = val_mask
    data['property'].test_mask = test_mask
    
    print(f"✅ Data splits: train={train_mask.sum().item()}, val={val_mask.sum().item()}, test={test_mask.sum().item()}")
    if use_stratify:
        print(f"   Cold-start in train: {cold_start[train_idx].sum()}, in test: {cold_start[test_idx].sum()}")
    
    return data

# Apply splits
data = prepare_data_splits(data, CONFIG['train_ratio'], CONFIG['val_ratio'])

In [ ]:
# Create model
model = create_model_from_data(
    data,
    hidden_dim=CONFIG['hidden_dim'],
    num_heads=CONFIG['num_heads'],
    num_layers=CONFIG['num_layers'],
)
model = model.to(DEVICE)

## 5. Training Loop (TPU-Optimized)

In [ ]:
def compute_metrics(predictions: torch.Tensor, targets: torch.Tensor) -> dict:
    """Compute evaluation metrics."""
    predictions = predictions.detach().cpu().numpy()
    targets = targets.detach().cpu().numpy()
    
    targets_safe = np.maximum(targets, 1.0)
    
    mape = np.mean(np.abs((predictions - targets) / targets_safe)) * 100
    mae = np.mean(np.abs(predictions - targets))
    rmse = np.sqrt(np.mean((predictions - targets) ** 2))
    
    ss_res = np.sum((targets - predictions) ** 2)
    ss_tot = np.sum((targets - np.mean(targets)) ** 2)
    r2 = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0.0
    
    return {'mape': mape, 'mae': mae, 'rmse': rmse, 'r2': r2}

In [ ]:
def move_data_to_device(data: HeteroData, device) -> tuple:
    """Move graph data to device."""
    x_dict = {}
    for k in data.node_types:
        if hasattr(data[k], 'x') and data[k].x is not None:
            x_dict[k] = data[k].x.to(device)
    
    edge_index_dict = {}
    for k in data.edge_types:
        if hasattr(data[k], 'edge_index') and data[k].edge_index is not None:
            edge_index_dict[k] = data[k].edge_index.to(device)
    
    targets = data['property'].y.to(device)
    train_mask = data['property'].train_mask.to(device)
    val_mask = data['property'].val_mask.to(device)
    
    cold_start_mask = None
    if hasattr(data['property'], 'cold_start_mask') and data['property'].cold_start_mask is not None:
        cold_start_mask = data['property'].cold_start_mask.to(device)
    
    return x_dict, edge_index_dict, targets, train_mask, val_mask, cold_start_mask

In [ ]:
def train_epoch_tpu(model, x_dict, edge_index_dict, targets, train_mask, cold_start_mask, optimizer, criterion):
    """Train for one epoch with XLA optimization."""
    model.train()
    optimizer.zero_grad()
    
    predictions = model(x_dict, edge_index_dict, cold_start_mask=cold_start_mask)
    
    # Get cold-start mask for training nodes only
    train_cold_mask = None
    if cold_start_mask is not None:
        train_cold_mask = cold_start_mask[train_mask]
    
    loss = criterion(
        predictions[train_mask],
        targets[train_mask],
        train_cold_mask,
    )
    
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    
    # XLA-specific: use xm.optimizer_step for proper sync
    if DEVICE_TYPE == 'tpu':
        xm.optimizer_step(optimizer, barrier=True)
    else:
        optimizer.step()
    
    return loss.item()


def evaluate_model(model, x_dict, edge_index_dict, targets, mask, cold_start_mask) -> dict:
    """Evaluate model on masked subset."""
    model.eval()
    
    with torch.no_grad():
        predictions = model(x_dict, edge_index_dict, cold_start_mask=cold_start_mask)
        
        # Sync for TPU before CPU transfer
        if DEVICE_TYPE == 'tpu':
            xm.mark_step()
        
        metrics = compute_metrics(predictions[mask], targets[mask])
    
    return metrics

In [ ]:
def train_model(model, data, config, device):
    """Full training loop with early stopping."""
    
    # Move data to device once
    print("Moving data to device...")
    x_dict, edge_index_dict, targets, train_mask, val_mask, cold_start_mask = move_data_to_device(data, device)
    print("✅ Data moved to device")
    
    criterion = ValuationLoss()
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config['learning_rate'],
        weight_decay=config['weight_decay'],
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=20, T_mult=2)
    
    history = {'train_loss': [], 'val_mape': [], 'val_r2': []}
    best_val_mape = float('inf')
    best_model_state = None
    epochs_without_improvement = 0
    
    pbar = tqdm(range(config['epochs']), desc="Training")
    
    for epoch in pbar:
        # Train
        train_loss = train_epoch_tpu(
            model, x_dict, edge_index_dict, targets,
            train_mask, cold_start_mask, optimizer, criterion
        )
        history['train_loss'].append(train_loss)
        
        # Evaluate every epoch
        val_metrics = evaluate_model(
            model, x_dict, edge_index_dict, targets, val_mask, cold_start_mask
        )
        history['val_mape'].append(val_metrics['mape'])
        history['val_r2'].append(val_metrics['r2'])
        
        scheduler.step()
        
        # Early stopping check
        if val_metrics['mape'] < best_val_mape:
            best_val_mape = val_metrics['mape']
            # Clone to CPU for safe storage
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
        
        pbar.set_postfix({
            'loss': f"{train_loss:.4f}",
            'val_mape': f"{val_metrics['mape']:.2f}%",
            'val_r2': f"{val_metrics['r2']:.4f}",
            'best': f"{best_val_mape:.2f}%",
        })
        
        if epochs_without_improvement >= config['patience']:
            print(f"\n⏹️ Early stopping at epoch {epoch + 1}")
            break
    
    # Load best model
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        model = model.to(device)
        print(f"✅ Loaded best model with Val MAPE: {best_val_mape:.2f}%")
    
    return model, history

In [ ]:
# Train the model
print(f"🚀 Starting training on {DEVICE_TYPE.upper()}...\n")

model, history = train_model(model, data, CONFIG, DEVICE)

## 6. Evaluation

In [ ]:
# Final evaluation on test set
x_dict, edge_index_dict, targets, _, _, cold_start_mask = move_data_to_device(data, DEVICE)
test_mask = data['property'].test_mask.to(DEVICE)

test_metrics = evaluate_model(model, x_dict, edge_index_dict, targets, test_mask, cold_start_mask)

print("\n📊 Test Set Results:")
print(f"   MAPE:  {test_metrics['mape']:.2f}%")
print(f"   MAE:   {test_metrics['mae']:,.0f} THB")
print(f"   RMSE:  {test_metrics['rmse']:,.0f} THB")
print(f"   R²:    {test_metrics['r2']:.4f}")

In [ ]:
# Cold-start vs Warm analysis
if hasattr(data['property'], 'cold_start_mask'):
    cold_start = data['property'].cold_start_mask.to(DEVICE)
    
    warm_mask = test_mask & ~cold_start
    cold_mask = test_mask & cold_start
    
    print("\n🌡️ Cold-Start Analysis:")
    
    if warm_mask.any():
        warm_metrics = evaluate_model(model, x_dict, edge_index_dict, targets, warm_mask, cold_start_mask)
        print(f"   Warm nodes (n={warm_mask.sum()}): MAPE={warm_metrics['mape']:.2f}%, R²={warm_metrics['r2']:.4f}")
    
    if cold_mask.any():
        cold_metrics = evaluate_model(model, x_dict, edge_index_dict, targets, cold_mask, cold_start_mask)
        print(f"   Cold-start (n={cold_mask.sum()}): MAPE={cold_metrics['mape']:.2f}%, R²={cold_metrics['r2']:.4f}")

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history['train_loss'])
axes[0].set_title('Training Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')

axes[1].plot(history['val_mape'])
axes[1].set_title('Validation MAPE (%)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAPE')

axes[2].plot(history['val_r2'])
axes[2].set_title('Validation R²')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('R²')

plt.tight_layout()
plt.show()

In [ ]:
# Predicted vs Actual scatter plot
model.eval()
with torch.no_grad():
    predictions = model(x_dict, edge_index_dict, cold_start_mask=cold_start_mask)
    if DEVICE_TYPE == 'tpu':
        xm.mark_step()

test_preds = predictions[test_mask].cpu().numpy()
test_targets = targets[test_mask].cpu().numpy()

plt.figure(figsize=(10, 10))
plt.scatter(test_targets / 1e6, test_preds / 1e6, alpha=0.5, s=10)
max_val = max(test_targets.max(), test_preds.max()) / 1e6
plt.plot([0, max_val], [0, max_val], 'r--', label='Perfect prediction')
plt.xlabel('Actual Price (Million THB)')
plt.ylabel('Predicted Price (Million THB)')
plt.title(f'HGT Property Valuation (MAPE: {test_metrics["mape"]:.2f}%)')
plt.legend()
plt.xlim(0, max_val * 1.1)
plt.ylim(0, max_val * 1.1)
plt.show()

## 7. Export Model

In [ ]:
# Save model locally
MODEL_DIR = Path('hgt_valuator')
MODEL_DIR.mkdir(exist_ok=True)

# Save model state dict (move to CPU first)
model_cpu = model.cpu()
torch.save(model_cpu.state_dict(), MODEL_DIR / 'hgt_valuator.pt')

# Save training history
with open(MODEL_DIR / 'training_history.json', 'w') as f:
    json.dump({k: [float(v) for v in vals] for k, vals in history.items()}, f, indent=2)

# Save metadata
metadata = {
    'config': CONFIG,
    'test_metrics': {k: float(v) for k, v in test_metrics.items()},
    'trained_at': datetime.now().isoformat(),
    'device': DEVICE_TYPE,
}
with open(MODEL_DIR / 'model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"✅ Model saved to {MODEL_DIR}")

In [ ]:
# Download trained model
from google.colab import files
import shutil

# Zip the model directory
shutil.make_archive('hgt_valuator', 'zip', MODEL_DIR)

print("📥 Downloading trained model...")
files.download('hgt_valuator.zip')

---

## 📋 Next Steps

1. **Unzip the model** on your local machine:
   ```bash
   unzip hgt_valuator.zip -d gis-server/models/hgt_valuator
   ```

2. **Start the API server**:
   ```bash
   cd gis-server
   uvicorn main:app --reload
   ```

3. **Test predictions**:
   ```bash
   curl -X POST http://localhost:8000/api/v1/hgt/predict \
     -H "Content-Type: application/json" \
     -d '{"lat": 13.7563, "lng": 100.5018, "area_sqm": 35, "bedrooms": 1}'
   ```